## Descriptive features embedding (caption + show_cover_text)

In [ ]:
# import torch
# from transformers import AutoTokenizer, AutoModel
# import polars as pl
# import glob
# import pyarrow as pa
# import pyarrow.parquet as pq

# print("Initializing Memory-Safe NLP Pipeline...")

# # 1. Initialize HuggingFace BGE Model
# MODEL_NAME = "BAAI/bge-small-zh-v1.5"
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# print(f"Loading Model {MODEL_NAME} onto {device}...")
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModel.from_pretrained(MODEL_NAME).to(device)
# model.eval()

# # 2. Fetch File Paths
# root = "/kaggle/input/datasets/nguyenngocanhle/kuairand-1k-parquet/kaggle/working/kuairand_parquet"

# def get_files(pattern):
#     files = glob.glob(f"{root}/{pattern}/**/*.parquet", recursive=True)
#     return [f for f in files if not f.endswith('.crc') and not f.endswith('_SUCCESS')]

# caption_files = get_files("kuairand_video_captions")
# video_files = get_files("video_features_basic_1k")

# if not caption_files or not video_files:
#     raise ValueError("🚨 Missing necessary parquet files!")

# # 3. Load & Filter Data (THE FIX)
# print("Loading valid video IDs from the 1K dataset...")
# # Extract only the ~4.3M videos actually present in your dataset
# valid_videos = pl.read_parquet(video_files).select("video_id").unique()

# print("Loading and filtering text data...")
# df_captions = pl.read_parquet(caption_files)

# # Inner join drops the 27+ million videos we don't need
# df_captions = df_captions.join(valid_videos, left_on="final_video_id", right_on="video_id", how="inner")

# df_captions = df_captions.with_columns(
#     pl.col("caption").fill_null(""),
#     pl.col("show_cover_text").fill_null("")
# ).with_columns(
#     (pl.col("caption") + " " + pl.col("show_cover_text")).alias("full_text")
# )

# video_ids = df_captions["final_video_id"].to_list()
# texts = df_captions["full_text"].to_list()
# print(f"Reduced target extraction to {len(texts)} relevant videos.")

# # 4. Extract and Write in Streaming Chunks (THE FIX)
# BATCH_SIZE = 256
# CHUNK_SIZE = 50000  # Flush to disk every 50k rows to save RAM

# # Define the Parquet schema for the output file
# schema = pa.schema([
#     ('video_id', pa.int64()),
#     ('nlp_vector', pa.list_(pa.float32()))
# ])

# output_path = "/kaggle/working/video_bge_embeddings.parquet"

# print(f"Extracting vectors in batches of {BATCH_SIZE}, writing every {CHUNK_SIZE}...")

# # Open a Parquet writer in streaming mode
# with torch.no_grad(), pq.ParquetWriter(output_path, schema) as writer:
#     chunk_vids = []
#     chunk_embs = []
    
#     for i in range(0, len(texts), BATCH_SIZE):
#         batch_texts = texts[i : i + BATCH_SIZE]
#         batch_vids = video_ids[i : i + BATCH_SIZE]
        
#         # Tokenize and push to GPU
#         inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        
#         # Forward pass & Extract [CLS]
#         outputs = model(**inputs)
#         cls_embeddings = outputs.last_hidden_state[:, 0, :]
        
#         # Normalize
#         cls_embeddings = torch.nn.functional.normalize(cls_embeddings, p=2, dim=1)
        
#         # Append to our temporary chunk
#         chunk_vids.extend(batch_vids)
#         chunk_embs.extend(cls_embeddings.cpu().numpy().tolist())
        
#         # If chunk is full, write to disk and clear RAM
#         if len(chunk_vids) >= CHUNK_SIZE or i + BATCH_SIZE >= len(texts):
#             table = pa.Table.from_arrays([chunk_vids, chunk_embs], schema=schema)
#             writer.write_table(table)
            
#             # Reset lists to free memory!
#             chunk_vids = []
#             chunk_embs = []
            
#         if (i % (BATCH_SIZE * 10)) == 0 and i > 0:
#             print(f"Processed {i}/{len(texts)}...")

# print(f"✅ NLP Extraction Complete! Saved safely to {output_path}")
print("Ran once")

## Load data and seperately save to different files due to limited free memory offered by Kaggle

In [1]:
!pip install polars

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 847.1/847.1 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 MB 99.6 MB/s eta 0:00:00:00:01

[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: pip install --upgrade pip


In [ ]:
# import polars as pl
# import time
# import glob

# print("Initializing Polars Lazy Pipeline for Two-Tower Retrieval...")
# start_time = time.time()

# root = "/kaggle/input/datasets/nguyenngocanhle/kuairand-1k-parquet/kaggle/working/kuairand_parquet"

# def get_clean_parquet_list(subfolder_pattern):
#     search_path = f"{root}/{subfolder_pattern}"
#     all_files = glob.glob(search_path, recursive=True)
#     clean_files = [f for f in all_files if not f.endswith('.crc') and not f.endswith('_SUCCESS')]
#     if not clean_files:
#         raise ValueError(f"🚨 No valid parquet files found for pattern: {subfolder_pattern}")
#     return clean_files

# # NOTE: video_features_statistic is explicitly omitted here for the Retrieval Stage
# log_files = get_clean_parquet_list("log_standard_*/**/*.parquet")
# user_files = get_clean_parquet_list("user_features_1k/**/*.parquet")
# video_files = get_clean_parquet_list("video_features_basic_1k/**/*.parquet")
# category_files = get_clean_parquet_list("kuairand_video_categories/**/*.parquet")
# embedding_file = ["/kaggle/input/datasets/nguyenngocanhle/video-bge-embeddings/kaggle/working/video_bge_embeddings.parquet"]

# # ==========================================
# # TABLE 1: Interactions Table (The Anchor)
# # ==========================================
# print("\n[1/3] Generating lightweight interactions.parquet...")
# (
#     pl.scan_parquet(log_files)
#     .filter(pl.col("is_click") == 1)
#     .select([
#         pl.col("user_id"),
#         pl.col("video_id").alias("target_video_id"),
#         pl.col("time_ms"),
#         pl.col("is_click")
#     ])
#     .sort("time_ms")
#     .collect(streaming=True)
#     .write_parquet('/kaggle/working/interactions.parquet')
# )

# # ==========================================
# # TABLE 2: User Table 
# # ==========================================
# print("\n[2/3] Generating user_table.parquet...")

# # 1. History Sequence
# seq_lazy = (
#     pl.scan_parquet(log_files)
#     .sort(["user_id", "time_ms"])
#     .group_by("user_id")
#     .agg(pl.col("video_id").tail(100).alias("history_sequence"))
# )

# # 2. Extract static user features + Binned Ranges
# user_feats = [
#     pl.col("user_active_degree").first(),
#     pl.col("is_live_streamer").first(),
#     pl.col("is_video_author").first(),
#     pl.col("follow_user_num_range").first(),
#     pl.col("fans_user_num_range").first(),
#     pl.col("friend_user_num_range").first(),
#     pl.col("register_days_range").first(),
# ] + [pl.col(f"onehot_feat{i}").first() for i in range(18)]

# users_lazy = (
#     pl.scan_parquet(user_files)
#     .group_by("user_id")
#     .agg(user_feats)
# )

# # 3. Join them together
# (
#     users_lazy
#     .join(seq_lazy, on="user_id", how="left")
#     # Fill users who have no prior clicks with an empty list
#     .with_columns(pl.col("history_sequence").fill_null([])) 
#     .collect(streaming=True)
#     .write_parquet('/kaggle/working/user_table.parquet')
# )

# # ==========================================
# # TABLE 3: Item Table
# # ==========================================
# print("\n[3/3] Generating item_table.parquet with NLP vectors...")

# # 1. Basic Metadata + Log-normalized duration
# videos_lazy = (
#     pl.scan_parquet(video_files)
#     .with_columns(
#         pl.col("video_duration").cast(pl.Float32).log1p().alias("duration_lognorm")
#     )
#     .group_by("video_id")
#     .agg([
#         pl.col("author_id").first(),
#         pl.col("video_type").first(),
#         pl.col("tag").first(),
#         pl.col("duration_lognorm").first()
#     ])
# )

# # 2. Extract Categories as Integer IDs instead of Strings
# cats_lazy = (
#     pl.scan_parquet(category_files)
#     .group_by("final_video_id")
#     .agg([
#         pl.col("first_level_category_id").first(),
#         pl.col("second_level_category_id").first()
#     ])
# )

# # 3. Text Vectors
# embeds_lazy = pl.scan_parquet(embedding_file)

# # 4. Chain the joins together
# (
#     videos_lazy
#     .join(cats_lazy, left_on="video_id", right_on="final_video_id", how="left")
#     .join(embeds_lazy, on="video_id", how="left")
#     .collect(streaming=True)
#     .write_parquet('/kaggle/working/item_table.parquet')
# )

# print(f"\n✅ Polars Pipeline Complete in {time.time() - start_time:.2f} seconds!")
print("Ran once")

In [7]:
!apt-get update && apt-get install -y zip

Get:1 http://deb.debian.org/debian trixie InRelease [140 kB]
Get:2 http://deb.debian.org/debian trixie-updates InRelease [47.3 kB]
Get:3 http://deb.debian.org/debian-security trixie-security InRelease [43.4 kB]
Get:4 http://deb.debian.org/debian trixie/main amd64 Packages [9673 kB]
Get:5 http://deb.debian.org/debian trixie-updates/main amd64 Packages.diff/Index [4995 B]
Err:5 http://deb.debian.org/debian trixie-updates/main amd64 Packages.diff/Index
  Need 4750 compressed bytes, but limit is 4412 and original is 4412
Get:6 http://deb.debian.org/debian-security trixie-security/main amd64 Packages [231 kB]
Get:5 http://deb.debian.org/debian trixie-updates/main amd64 Packages.diff/Index [4995 B]
Ign:5 http://deb.debian.org/debian trixie-updates/main amd64 Packages.diff/Index
Get:7 http://deb.debian.org/debian trixie-updates/main amd64 Packages [4412 B]
Fetched 10.1 MB in 1s (11.2 MB/s)                         
Reading package lists... Done
N: Repository 'http://deb.debian.org/debian trixi

In [8]:
import os
import subprocess
from IPython.display import FileLink, display

def download_file(path, download_file_name):
    os.chdir('/kaggle/working/')
    zip_name = f"/kaggle/working/{download_file_name}.zip"
    command = f"zip {zip_name} {path} -r"
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print("Unable to run zip command!")
        print(result.stderr)
        return
    display(FileLink(f'{download_file_name}.zip'))

# download_file("/kaggle/working/", "KuaiRand-1k-tables-parquet")
print("Ran once, download, and upload to kaggle Dataset")

Ran once, download, and upload to kaggle Dataset


In [2]:
import polars as pl

df_interactions = pl.read_parquet("/kaggle/input/datasets/nguyenngocanhle/two-tower-data/interactions.parquet")
df_users = pl.read_parquet("/kaggle/input/datasets/nguyenngocanhle/two-tower-data/user_table.parquet")
df_items = pl.read_parquet("/kaggle/input/datasets/nguyenngocanhle/two-tower-data/item_table.parquet")

print("=========================================")
print("Interactions")
print("==========================================")
print(df_interactions)

print("\n==========================================")
print("User features")
print("==========================================")
print(df_users)

print("\n==========================================")
print("Item features")
print("==========================================")
print(df_items)

Interactions
shape: (4_429_840, 4)
┌─────────┬─────────────────┬───────────────┬──────────┐
│ user_id ┆ target_video_id ┆ time_ms       ┆ is_click │
│ ---     ┆ ---             ┆ ---           ┆ ---      │
│ i32     ┆ i32             ┆ i64           ┆ i32      │
╞═════════╪═════════════════╪═══════════════╪══════════╡
│ 867     ┆ 1003341         ┆ 1649340003236 ┆ 1        │
│ 867     ┆ 993205          ┆ 1649340003236 ┆ 1        │
│ 867     ┆ 2711414         ┆ 1649340003236 ┆ 1        │
│ 867     ┆ 1657697         ┆ 1649340003236 ┆ 1        │
│ 867     ┆ 3517116         ┆ 1649340003236 ┆ 1        │
│ …       ┆ …               ┆ …             ┆ …        │
│ 490     ┆ 1900491         ┆ 1652025124592 ┆ 1        │
│ 490     ┆ 1414439         ┆ 1652025124592 ┆ 1        │
│ 346     ┆ 3602057         ┆ 1652025129022 ┆ 1        │
│ 346     ┆ 3377964         ┆ 1652025129022 ┆ 1        │
│ 346     ┆ 1408516         ┆ 1652025129022 ┆ 1        │
└─────────┴─────────────────┴───────────────┴────────

### Model defining and training (on TPU)

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import polars as pl
import numpy as np
import os

os.environ['PJRT_DEVICE'] = 'TPU' 
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as xla_pl

# ==========================================
# 1. THE TWO-TOWER ARCHITECTURE (FULL FEATURE SET)
# ==========================================

class UserTower(nn.Module):
    # num_dense_features = 25 (7 new features + 18 onehot features)
    def __init__(self, num_items, item_embed_dim=32, num_dense_features=25, final_dim=64):
        super().__init__()
        self.item_embedding = nn.Embedding(num_embeddings=num_items, embedding_dim=item_embed_dim, padding_idx=0)
        self.gru = nn.GRU(input_size=item_embed_dim, hidden_size=64, batch_first=True)
        self.static_mlp = nn.Sequential(nn.Linear(num_dense_features, 32), nn.ReLU())
        self.fusion_layer = nn.Linear(64 + 32, final_dim)

    def forward(self, history_seq, static_features):
        seq_emb = self.item_embedding(history_seq)
        _, hidden = self.gru(seq_emb)
        user_history_vector = hidden.squeeze(0) 
        user_static_vector = self.static_mlp(static_features) 
        combined = torch.cat([user_history_vector, user_static_vector], dim=1)
        return self.fusion_layer(combined)


class ItemTower(nn.Module):
    def __init__(self, num_items, num_categories=2000, num_tags=500, item_embed_dim=32, nlp_embed_dim=512, final_dim=64):
        super().__init__()
        self.item_embedding = nn.Embedding(num_items, item_embed_dim, padding_idx=0)
        
        # Categorical & Multi-Hot Tag Embeddings
        self.cat1_embedding = nn.Embedding(num_categories, 8, padding_idx=0)
        self.cat2_embedding = nn.Embedding(num_categories, 8, padding_idx=0)
        self.tag_embedding  = nn.Embedding(num_tags, 8, padding_idx=0)
        
        self.text_mlp = nn.Sequential(
            nn.Linear(nlp_embed_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 32)
        )
        
        # Fusion Input: 
        # 32 (Item ID) + 8 (Cat1) + 8 (Cat2) + 8 (Pooled Tags) + 32 (Text MLP) + 1 (Log Duration) = 89
        self.fusion_layer = nn.Linear(32 + 8 + 8 + 8 + 32 + 1, final_dim)

    def forward(self, item_id, cat1_id, cat2_id, tags, duration_log, precomputed_text_vector):
        id_emb = self.item_embedding(item_id)
        c1_emb = self.cat1_embedding(cat1_id)
        c2_emb = self.cat2_embedding(cat2_id)
        
        # Multi-hot Bag-of-Tags Pooling:
        # tags shape: (Batch, MAX_TAGS) -> tag_emb: (Batch, MAX_TAGS, 8)
        tag_emb = self.tag_embedding(tags)
        # Sum across MAX_TAGS dimension -> shape: (Batch, 8)
        tag_emb = tag_emb.sum(dim=1) 
        
        text_feat = self.text_mlp(precomputed_text_vector)
        dur_feat = duration_log.unsqueeze(1) 
        
        combined = torch.cat([id_emb, c1_emb, c2_emb, tag_emb, text_feat, dur_feat], dim=1)
        return self.fusion_layer(combined)


class TwoTowerModel(nn.Module):
    def __init__(self, num_items, num_categories, num_tags, final_dim=64):
        super().__init__()
        self.user_tower = UserTower(num_items=num_items, final_dim=final_dim)
        self.item_tower = ItemTower(
            num_items=num_items, 
            num_categories=num_categories, 
            num_tags=num_tags, 
            final_dim=final_dim
        )

    def forward(self, history_seq, user_static, item_id, cat1_id, cat2_id, tags, duration_log, item_features):
        u_vector = self.user_tower(history_seq, user_static)
        i_vector = self.item_tower(item_id, cat1_id, cat2_id, tags, duration_log, item_features)
        
        u_vector = torch.nn.functional.normalize(u_vector, p=2, dim=1)  
        i_vector = torch.nn.functional.normalize(i_vector, p=2, dim=1)  
        
        similarity_matrix = torch.matmul(u_vector, i_vector.T)
        return similarity_matrix * 10.0


# ==========================================
# 2. COMPLETE IN-BATCH DATASET (FIXED)
# ==========================================

class TwoTowerDataset(Dataset):
    def __init__(self, interactions_df, user_df, item_df, num_items, num_tags):
        self.user_ids = interactions_df['user_id'].to_numpy()
        self.target_ids = interactions_df['target_video_id'].to_numpy()
        self.num_tags = num_tags
        valid_item_ids = item_df['video_id'].to_numpy()

        # ==========================================
        # 1. MISSING BLOCK RESTORED: ITEM TENSORS
        # ==========================================
        print("Building Item Tensors in RAM...")
        
        # NLP Vector Tensor
        self.item_nlp_tensor = torch.zeros((num_items, 512), dtype=torch.float32)
        vecs = [v if v is not None else [0.0]*512 for v in item_df['nlp_vector'].to_list()]
        self.item_nlp_tensor[valid_item_ids] = torch.tensor(vecs, dtype=torch.float32)

        # Category Tensors
        self.cat1_tensor = torch.zeros(num_items, dtype=torch.long)
        c1_vals = np.maximum(0, item_df['first_level_category_id'].fill_null(0).to_numpy())
        self.cat1_tensor[valid_item_ids] = torch.tensor(c1_vals, dtype=torch.long)

        self.cat2_tensor = torch.zeros(num_items, dtype=torch.long)
        c2_vals = np.maximum(0, item_df['second_level_category_id'].fill_null(0).to_numpy())
        self.cat2_tensor[valid_item_ids] = torch.tensor(c2_vals, dtype=torch.long)

        # Log-Normalized Duration Tensor
        self.dur_tensor = torch.zeros(num_items, dtype=torch.float32)
        dur_vals = item_df['duration_lognorm'].fill_null(0.0).to_numpy()
        self.dur_tensor[valid_item_ids] = torch.tensor(dur_vals, dtype=torch.float32)

        # Multi-Value Tag Tensor Parsing
        MAX_TAGS = 5
        self.tag_tensor = torch.zeros((num_items, MAX_TAGS), dtype=torch.long)
        tag_strings = item_df['tag'].fill_null("").to_list()
        
        for idx, t_str in zip(valid_item_ids, tag_strings):
            if t_str and t_str != "UNKNOWN":
                t_ints = [int(x) for x in t_str.split(",") if x.isdigit()]
                if t_ints:
                    length = min(len(t_ints), MAX_TAGS)
                    self.tag_tensor[idx, :length] = torch.tensor(t_ints[:length], dtype=torch.long)

        # ==========================================
        # 2. USER TENSORS
        # ==========================================
        print("Building User Tensors in RAM...")
        max_user_id = user_df['user_id'].max() + 1
        valid_user_ids = user_df['user_id'].to_numpy()
        
        # Define explicit ordinal mappings 
        activity_map = {"unknown": 0.0, "low_active": 1.0, "high_active": 2.0, "full_active": 3.0}
        follow_map = {'0': 0.0, '(0,10]': 1.0, '(10,50]': 2.0, '(50,100]': 3.0, '(100,150]': 4.0, '(150,250]': 5.0, '(250,500]': 6.0, '500+': 7.0}
        fans_map = {'0': 0.0, '[1,10)': 1.0, '[10,100)': 2.0, '[100,1k)': 3.0, '[1k,5k)': 4.0, '[5k,1w)': 5.0, '[1w,10w)': 6.0}
        friend_map = {'0': 0.0, '[1,5)': 1.0, '[5,30)': 2.0, '[30,60)': 3.0, '[60,120)': 4.0, '[120,250)': 5.0, '250+': 6.0}
        register_map = {'15-30': 1.0, '31-60': 2.0, '61-90': 3.0, '91-180': 4.0, '181-365': 5.0, '366-730': 6.0, '730+': 7.0}

        # Apply mappings safely
        user_df = user_df.with_columns([
            pl.col("user_active_degree").replace(activity_map, default=0.0).cast(pl.Float32, strict=False),
            pl.col("follow_user_num_range").replace(follow_map, default=0.0).cast(pl.Float32, strict=False),
            pl.col("fans_user_num_range").replace(fans_map, default=0.0).cast(pl.Float32, strict=False),
            pl.col("friend_user_num_range").replace(friend_map, default=0.0).cast(pl.Float32, strict=False),
            pl.col("register_days_range").replace(register_map, default=0.0).cast(pl.Float32, strict=False),
            pl.col("is_live_streamer").cast(pl.Float32, strict=False),
            pl.col("is_video_author").cast(pl.Float32, strict=False)
        ])
        
        # Extract all 25 Static Columns 
        static_cols = [
            'user_active_degree', 'is_live_streamer', 'is_video_author',
            'follow_user_num_range', 'fans_user_num_range', 
            'friend_user_num_range', 'register_days_range'
        ] + [f'onehot_feat{i}' for i in range(18)]
        
        self.user_static_tensor = torch.zeros((max_user_id, 25), dtype=torch.float32)
        static_matrix = user_df.select(static_cols).fill_null(0.0).to_numpy()
        static_matrix = np.nan_to_num(static_matrix, nan=0.0) 
        self.user_static_tensor[valid_user_ids] = torch.tensor(static_matrix, dtype=torch.float32)

        # History Lookup
        self.user_history_lookup = {
            u: np.array(h, dtype=np.int64) if h is not None else np.array([], dtype=np.int64)
            for u, h in zip(valid_user_ids, user_df['history_sequence'].to_list())
        }

    def __len__(self):
        return len(self.user_ids) 

    def __getitem__(self, idx):
        u_id = self.user_ids[idx]
        i_id = self.target_ids[idx]

        raw_history = self.user_history_lookup.get(u_id, np.array([], dtype=np.int64))
        MAX_SEQ_LEN = 100
        padded_history = np.zeros(MAX_SEQ_LEN, dtype=np.int64)
        
        if len(raw_history) > 0:
            length = min(len(raw_history), MAX_SEQ_LEN)
            padded_history[-length:] = raw_history[-length:]
            
        return {
            "history_seq": torch.tensor(padded_history, dtype=torch.long),
            "user_static": self.user_static_tensor[u_id],
            "item_id": torch.tensor(i_id, dtype=torch.long),
            "cat1_id": self.cat1_tensor[i_id],
            "cat2_id": self.cat2_tensor[i_id],
            "tags": self.tag_tensor[i_id],
            "duration_log": self.dur_tensor[i_id],
            "item_features": self.item_nlp_tensor[i_id],
        }


# ==========================================
# 3. TRAINING & EVALUATION LOOP
# ==========================================

def evaluate(model, dataloader, device, k=10):
    model.eval()
    
    total_samples, total_recall, total_mrr = 0, 0, 0.0
    total_pos_logit, total_neg_logit = 0.0, 0.0
    
    with torch.no_grad():
        for batch in dataloader:
            history_seq = batch['history_seq']
            user_static = batch['user_static']
            item_id = batch['item_id']
            cat1 = batch['cat1_id']
            cat2 = batch['cat2_id']
            tags = batch['tags']
            dur = batch['duration_log']
            item_features = batch['item_features']
            
            logits = model(history_seq, user_static, item_id, cat1, cat2, tags, dur, item_features)
            batch_size = logits.size(0)
            
            if batch_size == 0:
                continue
                
            labels = torch.arange(batch_size, device=logits.device)
            
            # 1. Recall@K
            # Handle cases where batch_size < k
            k_val = min(k, batch_size)
            _, top_indices = torch.topk(logits, k=k_val, dim=1)
            total_recall += (top_indices == labels.unsqueeze(1)).sum().item()
            
            # 2. MRR
            ranks = (logits.argsort(dim=1, descending=True) == labels.unsqueeze(1)).nonzero(as_tuple=True)[1] + 1
            total_mrr += (1.0 / ranks.float()).sum().item()
            
            # 3. Positive Logits
            pos_logits = logits.diag()
            total_pos_logit += pos_logits.sum().item()
            
            # 4. Negative Logits
            if batch_size > 1:
                sum_all = logits.sum().item()
                sum_pos = pos_logits.sum().item()
                sum_neg = sum_all - sum_pos
                num_negatives = batch_size * (batch_size - 1)
                total_neg_logit += (sum_neg / num_negatives) * batch_size 
            
            total_samples += batch_size

    # SAFEGUARD: Prevent ZeroDivisionError if validation loader yields 0 samples
    if total_samples == 0:
        print("⚠️ Warning: Validation set yielded 0 samples! Check val_df length or batch_size.")
        return 0.0, 0.0, 0.0, 0.0

    return (
        total_recall / total_samples, 
        total_mrr / total_samples, 
        total_pos_logit / total_samples, 
        total_neg_logit / total_samples
    )


def train_two_tower_tpu(df_interactions, df_users, df_items, num_items, num_categories, num_tags):
    df_interactions = df_interactions.sort("time_ms")
    split_idx = int(len(df_interactions) * 0.8)
    train_df = df_interactions[:split_idx]
    val_df = df_interactions[split_idx:]
    
    train_dataset = TwoTowerDataset(train_df, df_users, df_items, num_items, num_tags)
    val_dataset = TwoTowerDataset(val_df, df_users, df_items, num_items, num_tags)
    
    # Use max tag index across train dataset
    num_tags = max(train_dataset.num_tags, 1)
    
    device = xm.xla_device()
    print(f"Training on device: {device}")

    train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True, num_workers=4, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=2048, shuffle=False, num_workers=4, drop_last=True)
    
    train_device_loader = xla_pl.MpDeviceLoader(train_loader, device)
    val_device_loader = xla_pl.MpDeviceLoader(val_loader, device)
    
    model = TwoTowerModel(
        num_items=num_items, 
        num_categories=num_categories, 
        num_tags=num_tags, 
        final_dim=64
    ).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    
    epochs = 10
    best_val_recall = 0.0
    patience = 2
    epochs_no_improve = 0 
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for batch_idx, batch in enumerate(train_device_loader):
            history_seq = batch['history_seq']
            user_static = batch['user_static']
            item_id = batch['item_id']
            cat1 = batch['cat1_id']
            cat2 = batch['cat2_id']
            tags = batch['tags']
            dur = batch['duration_log']
            item_features = batch['item_features']
            
            optimizer.zero_grad()
            logits = model(history_seq, user_static, item_id, cat1, cat2, tags, dur, item_features)
            labels = torch.arange(logits.size(0), device=logits.device)
            
            loss = criterion(logits, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            xm.optimizer_step(optimizer, barrier=True)
            total_loss += loss.item()
                
        val_recall, val_mrr, avg_pos, avg_neg = evaluate(model, val_device_loader, device, k=200) 

        print(f"--- Epoch {epoch+1} Completed")
        if batch_idx % 200 == 0:
            running_train_loss = total_train_loss / total_train_samples
        
            print(
                f"Batch [{batch_idx}/{len(train_loader)}] "
                f"Batch Loss: {loss.item():.4f} | "
                f"Avg Train Loss: {running_train_loss:.4f}"
            )
        
        print(f"📊 Val Recall@200: {val_recall:.4f} | MRR: {val_mrr:.4f} |  Avg Pos Logit: {avg_pos:.4f} | Avg Neg Logit: {avg_neg:.4f} ---")
        
        if val_recall > best_val_recall:
            best_val_recall = val_recall
            epochs_no_improve = 0 
            xm.save(model.state_dict(), "best_two_tower_inbatch.pth")
            print(f"🔥 New best model saved with Val Recall@10: {best_val_recall:.4f}")
        else:
            epochs_no_improve += 1
            print(f"⚠️ No improvement in Val Recall for {epochs_no_improve} epoch(s).")
            if epochs_no_improve >= patience:
                print(f"🛑 Early stopping triggered! Training halted to prevent overfitting.")
                break 
        
        print("-----------------------------------------")
        
    return model, device


# ==========================================
# 4. DATA EXECUTION BLOCK
# ==========================================
# df_interactions = pl.read_parquet("/kaggle/input/datasets/nguyenngocanhle/two-tower-data/interactions.parquet")
# df_users = pl.read_parquet("/kaggle/input/datasets/nguyenngocanhle/two-tower-data/user_table.parquet")
# df_items = pl.read_parquet("/kaggle/input/datasets/nguyenngocanhle/two-tower-data/item_table.parquet")

# 1. Calculate num_items
max_item_id = df_items["video_id"].max() or 0
max_int_id = df_interactions["target_video_id"].max() or 0
max_hist_id = df_users.select(pl.col("history_sequence").list.explode().max()).to_series()[0] or 0
num_items = int(max(max_item_id, max_int_id, max_hist_id)) + 1

# 2. Calculate num_categories
max_cat1 = df_items['first_level_category_id'].max() or 0
max_cat2 = df_items['second_level_category_id'].max() or 0
num_categories = int(max(max_cat1, max_cat2)) + 1

# 3. Calculate num_tags globally before dataset creation
tag_strings = df_items['tag'].fill_null("").to_list()
global_max_tag = 0
for t_str in tag_strings:
    if t_str and t_str != "UNKNOWN":
        t_ints = [int(x) for x in t_str.split(",") if x.isdigit()]
        if t_ints:
            global_max_tag = max(global_max_tag, max(t_ints))
num_tags = global_max_tag + 1

print(f"📊 Dataset Boundaries -> Items: {num_items:,} | Categories: {num_categories:,} | Tags: {num_tags:,}")

# Pass all 3 boundaries into training function
trained_model, device = train_two_tower_tpu(
    df_interactions=df_interactions, 
    df_users=df_users, 
    df_items=df_items, 
    num_items=num_items, 
    num_categories=num_categories, 
    num_tags=num_tags
)

/usr/local/lib/python3.12/site-packages/torch_xla/__init__.py:258: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(
/tmp/ipykernel_14/2351762500.py:369: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  max_hist_id = df_users.select(pl.col("history_sequence").list.explode().max()).to_series()[0] or 0


📊 Dataset Boundaries -> Items: 4,371,900 | Categories: 751 | Tags: 69
Building Item Tensors in RAM...
Building User Tensors in RAM...


/tmp/ipykernel_14/2351762500.py:158: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col("user_active_degree").replace(activity_map, default=0.0).cast(pl.Float32, strict=False),
/tmp/ipykernel_14/2351762500.py:159: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col("follow_user_num_range").replace(follow_map, default=0.0).cast(pl.Float32, strict=False),
/tmp/ipykernel_14/2351762500.py:160: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col("fans_user_num_range").replace(fans_map, default=0.0).cast(pl.Float32, strict=False),
/tmp/ipykernel_14/2351762500.py:161: DeprecationWarning: the `default` paramet

Building Item Tensors in RAM...
Building User Tensors in RAM...


/tmp/ipykernel_14/2351762500.py:290: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()
E0000 00:00:1785722946.552540      14 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:238


Training on device: xla:0


/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


--- Epoch 1 Complete | Val Recall@200: 0.2507 | MRR: 0.0116 ---
    ↳ Avg Pos Logit: 6.2924 | Avg Neg Logit: 5.6971
🔥 New best model saved with Val Recall@10: 0.2507
--- Epoch 2 Complete | Val Recall@200: 0.3158 | MRR: 0.0176 ---
    ↳ Avg Pos Logit: 6.5924 | Avg Neg Logit: 5.6815
🔥 New best model saved with Val Recall@10: 0.3158
--- Epoch 3 Complete | Val Recall@200: 0.3474 | MRR: 0.0211 ---
    ↳ Avg Pos Logit: 6.6536 | Avg Neg Logit: 5.5673
🔥 New best model saved with Val Recall@10: 0.3474
--- Epoch 4 Complete | Val Recall@200: 0.3573 | MRR: 0.0221 ---
    ↳ Avg Pos Logit: 6.3519 | Avg Neg Logit: 5.1048
🔥 New best model saved with Val Recall@10: 0.3573
--- Epoch 5 Complete | Val Recall@200: 0.3607 | MRR: 0.0223 ---
    ↳ Avg Pos Logit: 5.4556 | Avg Neg Logit: 4.1422
🔥 New best model saved with Val Recall@10: 0.3607
--- Epoch 6 Complete | Val Recall@200: 0.3636 | MRR: 0.0218 ---
    ↳ Avg Pos Logit: 6.2567 | Avg Neg Logit: 5.0130
🔥 New best model saved with Val Recall@10: 0.3636
--- 

In [9]:
download_file("/kaggle/working/best_two_tower_inbatch.pth", "best_two_tower_inbatch.pth")

/kaggle/working/best_two_tower_inbatch.pth.zip

## Extract item embedding

In [5]:
import torch
import torch.nn.functional as F
import polars as pl
import numpy as np
from torch.utils.data import Dataset, DataLoader

# Assuming torch_xla is set up
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as xla_pl

# ==========================================
# 1. ITEM EMBEDDING EXTRACTION 
# ==========================================

class ItemInferenceDataset(Dataset):
    def __init__(self, item_df, num_items):
        self.item_ids = item_df['video_id'].to_numpy()
        valid_item_ids = self.item_ids
        
        print("Building Inference Tensors in RAM...")
        
        # 1. NLP Tensor
        self.item_nlp_tensor = torch.zeros((num_items, 512), dtype=torch.float32)
        vecs = [v if v is not None else [0.0]*512 for v in item_df['nlp_vector'].to_list()]
        self.item_nlp_tensor[valid_item_ids] = torch.tensor(vecs, dtype=torch.float32)

        # 2. Categories
        self.cat1_tensor = torch.zeros(num_items, dtype=torch.long)
        c1_vals = np.maximum(0, item_df['first_level_category_id'].fill_null(0).to_numpy())
        self.cat1_tensor[valid_item_ids] = torch.tensor(c1_vals, dtype=torch.long)

        self.cat2_tensor = torch.zeros(num_items, dtype=torch.long)
        c2_vals = np.maximum(0, item_df['second_level_category_id'].fill_null(0).to_numpy())
        self.cat2_tensor[valid_item_ids] = torch.tensor(c2_vals, dtype=torch.long)

        # 3. Duration
        self.dur_tensor = torch.zeros(num_items, dtype=torch.float32)
        dur_vals = item_df['duration_lognorm'].fill_null(0.0).to_numpy()
        self.dur_tensor[valid_item_ids] = torch.tensor(dur_vals, dtype=torch.float32)

        # 4. Multi-hot Tags
        MAX_TAGS = 5
        self.tag_tensor = torch.zeros((num_items, MAX_TAGS), dtype=torch.long)
        tag_strings = item_df['tag'].fill_null("").to_list()
        
        for idx, t_str in zip(valid_item_ids, tag_strings):
            if t_str and t_str != "UNKNOWN":
                t_ints = [int(x) for x in t_str.split(",") if x.isdigit()]
                if t_ints:
                    length = min(len(t_ints), MAX_TAGS)
                    self.tag_tensor[idx, :length] = torch.tensor(t_ints[:length], dtype=torch.long)

    def __len__(self):
        return len(self.item_ids)

    def __getitem__(self, idx):
        i_id = self.item_ids[idx]
            
        return {
            "item_id": torch.tensor(i_id, dtype=torch.long),
            "cat1_id": self.cat1_tensor[i_id],
            "cat2_id": self.cat2_tensor[i_id],
            "tags": self.tag_tensor[i_id],
            "duration_log": self.dur_tensor[i_id],
            "item_features": self.item_nlp_tensor[i_id],
        }

def extract_and_save_item_embeddings(model, df_items, num_items, num_categories, num_tags, device):
    # model = TwoTowerModel(
    #     num_items=num_items, 
    #     num_categories=num_categories, 
    #     num_tags=num_tags, 
    #     final_dim=64
    # )
    
    # # 2. Load the Weights Dictionary safely into CPU memory first
    # state_dict = torch.load(model_path, map_location='cpu')
    
    # # 3. Inject weights into architecture and move to target device (GPU)
    # model.load_state_dict(state_dict)
    model.to(device)
    model.eval()

    print("Extracting Static Item Vectors...")
    dataset = ItemInferenceDataset(df_items, num_items)
    dataloader = DataLoader(dataset, batch_size=2048, shuffle=False, num_workers=4)
    
    device_loader = xla_pl.MpDeviceLoader(dataloader, device)
    
    all_video_ids, all_embeddings = [], []
    
    with torch.no_grad():
        for batch in device_loader:
            item_ids = batch['item_id']
            cat1 = batch['cat1_id']
            cat2 = batch['cat2_id']
            tags = batch['tags']
            dur = batch['duration_log']
            item_features = batch['item_features']
            
            # Pass features to item tower
            item_vectors = model.item_tower(item_ids, cat1, cat2, tags, dur, item_features)
            
            # L2 Normalize the vectors
            item_vectors = F.normalize(item_vectors, p=2, dim=1)
            
            if 'xla' in str(device):
                xm.mark_step()
                
            all_video_ids.extend(item_ids.cpu().numpy().tolist())
            all_embeddings.extend(item_vectors.cpu().numpy().tolist())

    df_item_faiss = pl.DataFrame({
        "video_id": all_video_ids, 
        "item_embedding_64d": all_embeddings
    })
    
    output_path = "/kaggle/working/faiss_item_embeddings.parquet"
    df_item_faiss.write_parquet(output_path)
    print(f"✅ Item Extraction Complete! Saved to {output_path}")
    
    return df_item_faiss


# ==========================================
# EXECUTION COMMANDS
# ==========================================
print("Extracting item embeddings for FAISS...")
df_item_faiss = extract_and_save_item_embeddings(
    model=trained_model, 
    df_items=df_items, 
    num_items=num_items, 
    num_categories=num_categories, 
    num_tags=num_tags, 
    device=device
)

Re-calculating Dictionary Boundaries for Model Initialization...


/tmp/ipykernel_14/4147293484.py:139: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  max_hist_id = df_users.select(pl.col("history_sequence").list.explode().max()).to_series()[0] or 0
/tmp/ipykernel_14/4147293484.py:158: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


Extracting item embeddings for FAISS...
Extracting Static Item Vectors...
Building Inference Tensors in RAM...


/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
/tmp/ipykernel_14/4147293484.py:108: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


✅ Item Extraction Complete! Saved to /kaggle/working/faiss_item_embeddings.parquet


In [9]:
download_file("/kaggle/working/faiss_item_embeddings.parquet","faiss_item_embeddings.parquet")

/kaggle/working/faiss_item_embeddings.parquet.zip